# Perfil dos dados de focos de calor

Este notebook responde, sobre os dados do INPE já carregados, as perguntas que podem mudar o projeto:

1. **Volume:** quantos focos há por mês e se cada arquivo traz só o seu mês.
2. **As 16 colunas:** vazios, `-999`, espaços sobrando e valores que não convertem para número.
3. **Unicidade do `id`:** se dá para deduplicar só pelo `id`.
4. **Satélites:** quais aparecem e como a tabela de órbitas classificou cada um.
5. **Focos sem FRP:** a parcela por satélite e por mês (o peso da justificativa 1 do modelo).
6. **Faixas e categorias:** coordenadas, risco de fogo, biomas, estados e acentos.
7. **Distribuição do FRP por órbita:** percentis nos meses de treino.
8. **Arquivos diários:** se o lado do streaming tem o mesmo formato e a mesma parcela sem FRP.

Ao final, o notebook gera um resumo em Markdown para colar em `docs/data_profile.md` e salva os gráficos para os slides. Cada execução fica numa pasta própria, com a data e a hora (UTC) no nome, em `profile_output/` e no bucket, em `evidencias/perfil/`.

**Pré-requisitos:** crawler rodado, Bronze carregada (`sql/bronze/01` e `02`) e tabelas de referência geradas depois dela (`sql/reference/`).

**Onde rodar:**
- **BigQuery Studio** (recomendado): no console, BigQuery > Notebook > Importar notebook. A autenticação já vem pronta.
- **Na máquina:** `pip install -r notebooks/requirements.txt` e `gcloud auth application-default login`. Rodando de dentro do repositório, o notebook lê os valores de `config.env` sozinho.

**Custo:** antes de cada consulta o notebook estima os bytes lidos (dry run) e se recusa a rodar acima de `LIMITE_GB_POR_CONSULTA`. A Bronze inteira tem poucos GB, bem dentro da cota gratuita de 1 TB por mês.

## 0. Configuração

Todas as constantes ficam nesta célula. Se `../config.env` existir (notebook aberto dentro do repositório), os valores vêm de lá; senão, valem os padrões abaixo, que devem ser iguais aos de `config.env`.

In [ ]:
import os
import re
from datetime import datetime, timezone
from typing import Any


def carregar_config_env(caminho) -> dict[Any, Any]:
    """Lê as linhas CHAVE=VALOR do config.env do repositório, expandindo ${VAR}."""
    valores = {}
    with open(caminho, encoding="utf-8") as arquivo:
        for linha in arquivo:
            linha = linha.strip()
            if not linha or linha.startswith("#") or "=" not in linha:
                continue
            chave, valor = linha.split("=", 1)
            valor = re.split(r"\s+#", valor, maxsplit=1)[0].strip().strip('"')
            valor = re.sub(r"\$\{(\w+)\}", lambda m: valores.get(m.group(1), ""), valor)
            valores[chave.strip()] = valor
    return valores


CAMINHO_CONFIG = "../config.env"
CONFIG = carregar_config_env(CAMINHO_CONFIG) if os.path.exists(CAMINHO_CONFIG) else {}

# =============================================================================
# Constantes (padrões iguais aos de config.env)
# =============================================================================
PROJECT_ID = CONFIG.get("PROJECT_ID", "inpe-fire-hotspots")
REGIAO = CONFIG.get("REGIAO", "us-central1")
DATASET = CONFIG.get("DATASET", "focos")
BUCKET = CONFIG.get("BUCKET", "inpe-fire-hotspots-data")
PASTA_RAW_DIARIO = CONFIG.get("PASTA_RAW_DIARIO", "raw/diario")
MES_FIM = CONFIG.get("MES_FIM", "2026-09")              # Último mês do histórico

TABELA_BRONZE = f"{PROJECT_ID}.{DATASET}.bronze_focos_mensal"
TABELA_SATELITES = f"{PROJECT_ID}.{DATASET}.satelites"   # Gerada da Bronze por sql/reference/01_satelites.sql
URIS_DIARIOS = f"gs://{BUCKET}/{PASTA_RAW_DIARIO}/*.csv"
ENCODING_CSV = "UTF-8"                                   # O mesmo de sql/bronze/01_external_tables.sql

MES_FIM_TREINO = "2026-08"     # Meses de treino até essa data
LIMITE_GB_POR_CONSULTA = 5     # Nenhuma consulta roda se for ler mais do que isso
EXECUCAO = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S")   # Nome da pasta desta execução (UTC)
PASTA_SAIDA = os.path.join("profile_output", EXECUCAO)   # Gráficos (PNG) e resumo (Markdown)
ENVIAR_PARA_BUCKET = True      # Copia as saídas para o bucket no fim
PASTA_BUCKET_SAIDA = f"evidencias/perfil/{EXECUCAO}"

FORMATO_DATA_INPE = "%Y-%m-%d %H:%M:%S"
VALOR_AUSENTE = -999
LAT_MIN, LAT_MAX = -34.0, 5.5   # Retângulo do Brasil, o mesmo da validação do streaming
LON_MIN, LON_MAX = -74.5, -28.5
PERCENTIS_FRP = [50, 75, 90, 95, 99]
PERCENTIL_LIMIAR = 90           # Proposta do plano para "foco intenso"
BINS_POR_DECADA = 10            # Resolução do histograma do FRP em escala log
QTD_ESTADOS_ESPERADA = 27

COLUNAS_INPE = [
    "id", "lat", "lon", "data_hora_gmt", "satelite", "municipio", "estado", "pais",
    "municipio_id", "estado_id", "pais_id", "numero_dias_sem_chuva", "precipitacao",
    "risco_fogo", "bioma", "frp",
]
COLUNAS_NUMERICAS = [
    "lat", "lon", "municipio_id", "estado_id", "pais_id",
    "numero_dias_sem_chuva", "precipitacao", "risco_fogo", "frp",
]
COLUNAS_CONTROLE_BRONZE = ["nome_arquivo", "mes_referencia", "carregado_em"]

# Texto com acento quebrado: caractere de substituição ou "Ã" seguido de outro caractere latino
PADRAO_ACENTO_QUEBRADO = r"\x{FFFD}|Ã[\x{0080}-\x{00BF}]"

# Cores dos gráficos
COR_PRINCIPAL = "#2a78d6"
COR_POR_ORBITA = {"geoestacionario": "#2a78d6", "polar": "#eb6834", "desconhecida": "#a3a29c"}
COR_REFERENCIA = "#52514e"
# =============================================================================

print(f"Projeto: {PROJECT_ID} | região: {REGIAO} | Bronze: {TABELA_BRONZE}")
print(f"Diários: {URIS_DIARIOS} | treino até {MES_FIM_TREINO}")
print(f"Saídas desta execução: {PASTA_SAIDA}" + (f" e gs://{BUCKET}/{PASTA_BUCKET_SAIDA}/" if ENVIAR_PARA_BUCKET else ""))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from google.cloud import bigquery, storage

cliente = bigquery.Client(project=PROJECT_ID, location=REGIAO)
os.makedirs(PASTA_SAIDA, exist_ok=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

BYTES_LIDOS = {"total": 0}
RESUMO = {}   # Números-chave, preenchidos ao longo do notebook

plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ae",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e6e5e0",
    "grid.linewidth": 0.6,
})


def consultar(sql, parametros=None, tabelas_externas=None) -> Any:
    """Roda uma consulta com estimativa de custo antes (dry run) e limite de bytes."""
    limite = int(LIMITE_GB_POR_CONSULTA * 1024**3)

    def configuracao(dry_run) -> bigquery.QueryJobConfig:
        config = bigquery.QueryJobConfig(
            dry_run=dry_run,
            use_query_cache=not dry_run,
            query_parameters=parametros or [],
            table_definitions=tabelas_externas or {},
        )
        if not dry_run:
            config.maximum_bytes_billed = limite
        return config

    estimativa = cliente.query(sql, job_config=configuracao(True)).total_bytes_processed or 0
    if estimativa > limite:
        raise RuntimeError(
            f"Consulta leria {estimativa / 1024**3:.2f} GB, acima do limite de {LIMITE_GB_POR_CONSULTA} GB."
        )
    job = cliente.query(sql, job_config=configuracao(False))
    resultado = job.to_dataframe()
    lidos = job.total_bytes_processed or 0
    BYTES_LIDOS["total"] += lidos
    print(f"[{lidos / 1024**2:,.1f} MB lidos, {BYTES_LIDOS['total'] / 1024**3:,.2f} GB no notebook]")
    return resultado


def ausente(coluna) -> str:
    """Regra de ausência do projeto: vazio ou -999, depois de aparar espaços."""
    return f"(TRIM(IFNULL({coluna}, '')) = '' OR SAFE_CAST(TRIM({coluna}) AS FLOAT64) = {VALOR_AUSENTE})"


def numero(coluna) -> str:
    """Texto para FLOAT64, com vazio e -999 virando NULL."""
    return f"IF({ausente(coluna)}, NULL, SAFE_CAST(TRIM({coluna}) AS FLOAT64))"


def sql_orbitas() -> str:
    return f"SELECT satelite, tipo_orbita, max_horas_num_dia FROM `{TABELA_SATELITES}`"


def salvar(figura, nome) -> None:
    caminho = os.path.join(PASTA_SAIDA, f"{nome}.png")
    figura.savefig(caminho, dpi=150, bbox_inches="tight")
    print(f"Gráfico salvo em {caminho}")


def milhar(valor, _posicao=None) -> str:
    return f"{valor:,.0f}".replace(",", ".")


tabela = cliente.get_table(TABELA_BRONZE)
print(f"Bronze: {milhar(tabela.num_rows)} linhas, {tabela.num_bytes / 1024**3:.2f} GB")

## 1. Volume por mês

Cada arquivo mensal deve trazer só focos do seu mês. `fora_do_mes` diferente de zero indica arquivo com dias de outro mês, o que pode duplicar focos entre arquivos.

In [ ]:
DATA_HORA = f"SAFE.PARSE_TIMESTAMP('{FORMATO_DATA_INPE}', TRIM(data_hora_gmt))"

por_mes = consultar(f"""
SELECT
  mes_referencia,
  COUNT(DISTINCT nome_arquivo) AS arquivos,
  COUNT(*) AS focos,
  COUNTIF({ausente('frp')}) AS sem_frp,
  ROUND(100 * COUNTIF({ausente('frp')}) / COUNT(*), 1) AS pct_sem_frp,
  MIN({DATA_HORA}) AS primeira_deteccao,
  MAX({DATA_HORA}) AS ultima_deteccao,
  COUNTIF(DATE_TRUNC(DATE({DATA_HORA}), MONTH) != mes_referencia) AS fora_do_mes
FROM `{TABELA_BRONZE}`
GROUP BY mes_referencia
ORDER BY mes_referencia
""")

RESUMO["total_focos"] = int(por_mes["focos"].sum())
RESUMO["meses"] = len(por_mes)
RESUMO["fora_do_mes"] = int(por_mes["fora_do_mes"].sum())
RESUMO["pct_sem_frp_geral"] = round(100 * por_mes["sem_frp"].sum() / por_mes["focos"].sum(), 1)
por_mes

In [ ]:
figura, eixo = plt.subplots(figsize=(10, 4))
rotulos = pd.to_datetime(por_mes["mes_referencia"].astype(str)).dt.strftime("%m/%y")
eixo.bar(rotulos, por_mes["focos"], color=COR_PRINCIPAL, width=0.7)
eixo.yaxis.set_major_formatter(mticker.FuncFormatter(milhar))
eixo.grid(axis="x", visible=False)
eixo.set_title("Focos de calor por mês (arquivos mensais do INPE)")
eixo.set_ylabel("focos")
pico = por_mes["focos"].idxmax()
eixo.annotate(milhar(por_mes.loc[pico, "focos"]), (pico, por_mes.loc[pico, "focos"]),
              ha="center", va="bottom", xytext=(0, 3), textcoords="offset points")
salvar(figura, "01_focos_por_mes")
plt.show()

## 2. As 16 colunas

Confere se a Bronze tem as 16 colunas do INPE na ordem esperada e, para cada uma, quantos valores estão vazios, valem `-999`, têm espaços sobrando ou não convertem para número.

In [ ]:
colunas_bronze = [campo.name for campo in tabela.schema]
esperadas = COLUNAS_INPE + COLUNAS_CONTROLE_BRONZE
if colunas_bronze == esperadas:
    print("OK: as 16 colunas do INPE estão na ordem esperada, mais as 3 de controle.")
else:
    print("ATENÇÃO: colunas diferentes do esperado")
    print("  faltando:", [c for c in esperadas if c not in colunas_bronze])
    print("  sobrando:", [c for c in colunas_bronze if c not in esperadas])

partes = []
for coluna in COLUNAS_INPE:
    partes.append(f"COUNTIF(TRIM(IFNULL({coluna}, '')) = '') AS {coluna}__vazio")
    partes.append(f"COUNTIF({coluna} != TRIM({coluna})) AS {coluna}__espacos")
    if coluna in COLUNAS_NUMERICAS:
        partes.append(f"COUNTIF(SAFE_CAST(TRIM({coluna}) AS FLOAT64) = {VALOR_AUSENTE}) AS {coluna}__menos999")
        partes.append(
            f"COUNTIF(TRIM(IFNULL({coluna}, '')) != '' AND SAFE_CAST(TRIM({coluna}) AS FLOAT64) IS NULL)"
            f" AS {coluna}__nao_numerico"
        )
qualidade_bruta = consultar(f"SELECT COUNT(*) AS total, {', '.join(partes)} FROM `{TABELA_BRONZE}`").iloc[0]

total = int(qualidade_bruta["total"])
qualidade = pd.DataFrame([
    {
        "coluna": coluna,
        "vazio": int(qualidade_bruta[f"{coluna}__vazio"]),
        "menos_999": int(qualidade_bruta.get(f"{coluna}__menos999", 0)),
        "nao_numerico": int(qualidade_bruta.get(f"{coluna}__nao_numerico", 0)),
        "com_espacos": int(qualidade_bruta[f"{coluna}__espacos"]),
    }
    for coluna in COLUNAS_INPE
])
qualidade["pct_ausente"] = (100 * (qualidade["vazio"] + qualidade["menos_999"]) / total).round(2)
RESUMO["colunas_com_ausentes"] = qualidade.loc[qualidade["pct_ausente"] > 0, ["coluna", "pct_ausente"]].values.tolist()
RESUMO["colunas_nao_numericas"] = qualidade.loc[qualidade["nao_numerico"] > 0, "coluna"].tolist()
RESUMO["colunas_com_espacos"] = qualidade.loc[qualidade["com_espacos"] > 0, "coluna"].tolist()
qualidade

In [ ]:
com_ausentes = qualidade[qualidade["pct_ausente"] > 0].sort_values("pct_ausente")
if com_ausentes.empty:
    print("Nenhuma coluna com valores ausentes.")
else:
    figura, eixo = plt.subplots(figsize=(8, 0.45 * len(com_ausentes) + 1))
    eixo.barh(com_ausentes["coluna"], com_ausentes["pct_ausente"], color=COR_PRINCIPAL, height=0.6)
    for posicao, valor in enumerate(com_ausentes["pct_ausente"]):
        eixo.annotate(f"{valor:.1f}%", (valor, posicao), va="center", xytext=(4, 0), textcoords="offset points")
    eixo.grid(axis="y", visible=False)
    eixo.set_xlabel("% de focos com valor vazio ou -999")
    eixo.set_title("Valores ausentes por coluna")
    salvar(figura, "02_ausentes_por_coluna")
    plt.show()

## 3. Unicidade do `id`

O plano deduplica pelo `id`. Se ele repetir, vale olhar se as repetições são linhas idênticas (o mesmo foco em dois arquivos) ou focos diferentes com o mesmo `id`; neste caso a chave passa a ser satélite, horário, latitude e longitude.

In [ ]:
linha_inteira = ", ".join(f"TRIM({c})" for c in COLUNAS_INPE)
unicidade = consultar(f"""
SELECT
  COUNT(*) AS linhas,
  COUNT(DISTINCT TRIM(id)) AS ids_distintos,
  COUNT(DISTINCT CONCAT(TRIM(satelite), '|', TRIM(data_hora_gmt), '|', TRIM(lat), '|', TRIM(lon))) AS chaves_alternativas,
  COUNT(DISTINCT TO_JSON_STRING(STRUCT({linha_inteira}))) AS linhas_distintas
FROM `{TABELA_BRONZE}`
""").iloc[0]

repetidos = int(unicidade["linhas"] - unicidade["ids_distintos"])
RESUMO["ids_repetidos"] = repetidos
if repetidos == 0:
    RESUMO["conclusao_id"] = "O id é único: a deduplicação pode usar só o id."
elif unicidade["linhas_distintas"] == unicidade["ids_distintos"]:
    RESUMO["conclusao_id"] = (f"{repetidos} linhas repetem o id, mas são cópias idênticas: "
                              "deduplicar pelo id resolve.")
else:
    RESUMO["conclusao_id"] = (f"{repetidos} linhas repetem o id com conteúdo diferente: "
                              "usar a chave satélite, horário, lat e lon.")
print(unicidade.to_string())
print("\n" + RESUMO["conclusao_id"])

In [ ]:
if repetidos:
    exemplos = consultar(f"""
    SELECT * EXCEPT (carregado_em)
    FROM `{TABELA_BRONZE}`
    WHERE TRIM(id) IN (
      SELECT TRIM(id) FROM `{TABELA_BRONZE}` GROUP BY 1 HAVING COUNT(*) > 1 LIMIT 5
    )
    ORDER BY id, nome_arquivo
    """)
    display(exemplos)

## 4. Satélites e órbitas

A tabela `satelites` é gerada da Bronze por `sql/reference/01_satelites.sql`: a órbita sai da quantidade de horas diferentes com focos no dia de maior atividade de cada satélite (`max_horas_num_dia`). Um geoestacionário detecta o dia todo; um polar, só nas poucas passagens sobre o Brasil. Esta seção confere a classificação. `desconhecida` só aparece se a tabela não foi gerada de novo depois da última carga da Bronze.

In [ ]:
satelites = consultar(f"""
WITH orbitas AS ({sql_orbitas()})
SELECT
  TRIM(b.satelite) AS satelite,
  IFNULL(o.tipo_orbita, 'desconhecida') AS tipo_orbita,
  o.max_horas_num_dia,
  COUNT(*) AS focos,
  COUNTIF({ausente('b.frp')}) AS sem_frp,
  ROUND(100 * COUNTIF({ausente('b.frp')}) / COUNT(*), 1) AS pct_sem_frp,
  MIN(b.mes_referencia) AS primeiro_mes,
  MAX(b.mes_referencia) AS ultimo_mes
FROM `{TABELA_BRONZE}` AS b
LEFT JOIN orbitas AS o ON TRIM(b.satelite) = o.satelite
GROUP BY 1, 2, 3
ORDER BY focos DESC
""")

desconhecidos = satelites.loc[satelites["tipo_orbita"] == "desconhecida", "satelite"].tolist()
sem_frp_total = satelites.loc[satelites["pct_sem_frp"] == 100, "satelite"].tolist()
RESUMO["satelites"] = satelites["satelite"].tolist()
RESUMO["satelites_desconhecidos"] = desconhecidos
RESUMO["satelites_nunca_tem_frp"] = sem_frp_total
satelites

In [ ]:
if desconhecidos:
    print("Satélites sem órbita:", desconhecidos)
    print("Rode de novo: bash sql/run_sql.sh sql/reference/01_satelites.sql")
else:
    print("Todos os satélites da Bronze estão na tabela de órbitas.")
print(satelites.groupby("tipo_orbita")["max_horas_num_dia"].agg(["min", "max"]).to_string())

print("\nSatélites que nunca trazem FRP (sem rótulo, o modelo não aprende com eles):", sem_frp_total or "nenhum")

In [ ]:
ordenado = satelites.sort_values("pct_sem_frp")
figura, eixo = plt.subplots(figsize=(8, 0.45 * len(ordenado) + 1.2))
cores = [COR_POR_ORBITA[o] for o in ordenado["tipo_orbita"]]
eixo.barh(ordenado["satelite"], ordenado["pct_sem_frp"], color=cores, height=0.6)
for posicao, (pct, focos) in enumerate(zip(ordenado["pct_sem_frp"], ordenado["focos"])):
    eixo.annotate(f"{pct:.0f}%  ({milhar(focos)} focos)", (pct, posicao), va="center",
                  xytext=(4, 0), textcoords="offset points")
eixo.set_xlim(0, 125)
eixo.xaxis.set_major_locator(mticker.FixedLocator([0, 25, 50, 75, 100]))
eixo.grid(axis="y", visible=False)
eixo.set_xlabel("% dos focos sem FRP")
eixo.set_title("Focos sem FRP por satélite")
legenda = [plt.Rectangle((0, 0), 1, 1, color=c) for o, c in COR_POR_ORBITA.items() if o in set(ordenado["tipo_orbita"])]
eixo.legend(legenda, [o for o in COR_POR_ORBITA if o in set(ordenado["tipo_orbita"])],
            title="órbita", frameon=False, loc="lower right")
salvar(figura, "04_sem_frp_por_satelite")
plt.show()

## 5. Focos sem FRP por mês e satélite

A parcela de focos sem FRP é o peso da justificativa 1 do modelo (estimar a intensidade quando o satélite não mede). Se ela variar muito entre meses, o teste em setembro pode não representar o streaming de outubro em diante.

In [ ]:
mes_satelite = consultar(f"""
SELECT
  mes_referencia,
  TRIM(satelite) AS satelite,
  COUNT(*) AS focos,
  ROUND(100 * COUNTIF({ausente('frp')}) / COUNT(*), 1) AS pct_sem_frp
FROM `{TABELA_BRONZE}`
GROUP BY 1, 2
""")
grade = mes_satelite.pivot(index="satelite", columns="mes_referencia", values="pct_sem_frp")
grade = grade.reindex(satelites["satelite"])   # Mesma ordem da seção 4 (mais focos primeiro)

figura, eixo = plt.subplots(figsize=(11, 0.5 * len(grade) + 1.5))
imagem = eixo.imshow(np.ma.masked_invalid(grade.values), cmap="Blues", vmin=0, vmax=100, aspect="auto")
eixo.set_xticks(range(grade.shape[1]), pd.to_datetime(grade.columns.astype(str)).strftime("%m/%y"))
eixo.set_yticks(range(grade.shape[0]), grade.index)
eixo.grid(False)
for i in range(grade.shape[0]):
    for j in range(grade.shape[1]):
        valor = grade.values[i, j]
        if not np.isnan(valor):
            eixo.text(j, i, f"{valor:.0f}", ha="center", va="center", fontsize=8,
                      color="white" if valor > 60 else "#0b0b0b")
figura.colorbar(imagem, ax=eixo, label="% sem FRP", shrink=0.8)
eixo.set_title("% de focos sem FRP por satélite e mês (célula vazia: satélite sem focos no mês)")
salvar(figura, "05_sem_frp_mes_satelite")
plt.show()

print(f"Parcela geral sem FRP: {RESUMO['pct_sem_frp_geral']}%")
print(por_mes[["mes_referencia", "focos", "pct_sem_frp"]].to_string(index=False))

## 6. Faixas, datas e categorias

Estima quanto o streaming rejeitaria pelas mesmas regras da validação (`pipeline/focos_stream/cleaning.py`) e lista os valores das colunas de texto, para conferir acentos e grafias.

In [ ]:
faixas = consultar(f"""
WITH valores AS (
  SELECT
    {DATA_HORA} AS data_hora,
    SAFE_CAST(TRIM(lat) AS FLOAT64) AS lat,
    SAFE_CAST(TRIM(lon) AS FLOAT64) AS lon,
    {numero('risco_fogo')} AS risco_fogo,
    {numero('precipitacao')} AS precipitacao,
    {numero('numero_dias_sem_chuva')} AS dias_sem_chuva,
    {numero('frp')} AS frp,
    TRIM(pais) AS pais
  FROM `{TABELA_BRONZE}`
)
SELECT
  COUNT(*) AS total,
  COUNTIF(data_hora IS NULL) AS data_hora_invalida,
  COUNTIF(lat IS NULL OR lon IS NULL) AS coordenada_invalida,
  COUNTIF(lat NOT BETWEEN {LAT_MIN} AND {LAT_MAX} OR lon NOT BETWEEN {LON_MIN} AND {LON_MAX}) AS fora_do_brasil,
  COUNTIF(risco_fogo NOT BETWEEN 0 AND 1) AS risco_fogo_fora_de_0_a_1,
  COUNTIF(precipitacao < 0) AS precipitacao_negativa,
  COUNTIF(dias_sem_chuva < 0) AS dias_sem_chuva_negativo,
  COUNTIF(frp < 0) AS frp_negativo,
  COUNTIF(frp = 0) AS frp_zero,
  COUNTIF(pais != 'Brasil') AS pais_diferente_de_brasil
FROM valores
""").iloc[0]

faixas_tabela = faixas.drop("total").rename("focos").to_frame()
faixas_tabela["pct"] = (100 * faixas_tabela["focos"] / faixas["total"]).round(3)
rejeitaria = int(faixas[["data_hora_invalida", "coordenada_invalida", "fora_do_brasil", "risco_fogo_fora_de_0_a_1"]].sum())
RESUMO["pct_rejeicao_estimada"] = round(100 * rejeitaria / faixas["total"], 3)
print(f"Rejeição estimada pelas regras do streaming: no máximo {RESUMO['pct_rejeicao_estimada']}% dos focos")
faixas_tabela

In [ ]:
categorias = consultar(f"""
SELECT 'bioma' AS coluna, TRIM(bioma) AS valor, COUNT(*) AS focos FROM `{TABELA_BRONZE}` GROUP BY 2
UNION ALL
SELECT 'estado', TRIM(estado), COUNT(*) FROM `{TABELA_BRONZE}` GROUP BY 2
UNION ALL
SELECT 'pais', TRIM(pais), COUNT(*) FROM `{TABELA_BRONZE}` GROUP BY 2
ORDER BY coluna, focos DESC
""")
RESUMO["biomas"] = categorias.loc[categorias["coluna"] == "bioma", "valor"].tolist()
RESUMO["qtd_estados"] = int((categorias["coluna"] == "estado").sum())
print(f"Estados distintos: {RESUMO['qtd_estados']} (esperado: {QTD_ESTADOS_ESPERADA})")
print("Biomas:", RESUMO["biomas"])
print("Países:", categorias.loc[categorias["coluna"] == "pais", "valor"].tolist())

acentos = consultar(f"""
SELECT
  COUNTIF(REGEXP_CONTAINS(CONCAT(IFNULL(municipio, ''), IFNULL(estado, ''), IFNULL(bioma, '')), @padrao)) AS suspeitos,
  ARRAY_AGG(DISTINCT IF(REGEXP_CONTAINS(municipio, @padrao), municipio, NULL) IGNORE NULLS LIMIT 10) AS exemplos
FROM `{TABELA_BRONZE}`
""", parametros=[bigquery.ScalarQueryParameter("padrao", "STRING", PADRAO_ACENTO_QUEBRADO)]).iloc[0]
RESUMO["acentos_suspeitos"] = int(acentos["suspeitos"])
if acentos["suspeitos"]:
    print(f"\nATENÇÃO: {acentos['suspeitos']} linhas com acento quebrado. Exemplos: {list(acentos['exemplos'])}")
    print("Troque ENCODING_CSV / encoding em sql/bronze/01_external_tables.sql para 'ISO-8859-1' e recarregue.")
else:
    print("\nAcentos OK: nenhum texto com codificação quebrada.")

## 7. Distribuição do FRP por tipo de órbita

Só nos meses de treino (até `MES_FIM_TREINO`), para que o mês de teste não influencie o limiar. O valor final do limiar é decisão dela.

In [ ]:
FRP_TREINO = f"""
WITH orbitas AS ({sql_orbitas()})
SELECT o.tipo_orbita, TRIM(b.satelite) AS satelite, {numero('b.frp')} AS frp
FROM `{TABELA_BRONZE}` AS b
JOIN orbitas AS o ON TRIM(b.satelite) = o.satelite
WHERE b.mes_referencia <= DATE '{MES_FIM_TREINO}-01'
  AND NOT {ausente('b.frp')}
"""

quantis = consultar(f"""
WITH focos_frp AS ({FRP_TREINO})
SELECT tipo_orbita, '(todos)' AS satelite, COUNT(*) AS focos_com_frp,
       COUNTIF(frp <= 0) AS frp_nao_positivo, APPROX_QUANTILES(frp, 100) AS q
FROM focos_frp GROUP BY 1
UNION ALL
SELECT tipo_orbita, satelite, COUNT(*), COUNTIF(frp <= 0), APPROX_QUANTILES(frp, 100)
FROM focos_frp GROUP BY 1, 2
ORDER BY tipo_orbita, satelite
""")
for p in PERCENTIS_FRP:
    quantis[f"p{p}"] = quantis["q"].apply(lambda q, p=p: round(float(q[p]), 1))
quantis = quantis.drop(columns="q")

limiares = quantis[quantis["satelite"] == "(todos)"].set_index("tipo_orbita")[f"p{PERCENTIL_LIMIAR}"]
RESUMO["limiares_candidatos"] = limiares.to_dict()
print(f"Limiar candidato (P{PERCENTIL_LIMIAR} dos meses de treino), em MW:")
for orbita, valor in limiares.items():
    print(f"  {orbita}: {valor}")
quantis

In [ ]:
histograma = consultar(f"""
WITH focos_frp AS ({FRP_TREINO})
SELECT tipo_orbita,
       FLOOR(LOG10(frp) * {BINS_POR_DECADA}) / {BINS_POR_DECADA} AS log10_frp,
       COUNT(*) AS focos
FROM focos_frp
WHERE frp > 0
GROUP BY 1, 2
ORDER BY 1, 2
""")

orbitas = list(limiares.index)
figura, eixos = plt.subplots(1, len(orbitas), figsize=(5.5 * len(orbitas), 3.8), sharey=False, squeeze=False)
for eixo, orbita in zip(eixos[0], orbitas):
    dados = histograma[histograma["tipo_orbita"] == orbita]
    eixo.bar(dados["log10_frp"], dados["focos"], width=1 / BINS_POR_DECADA * 0.9, align="edge",
             color=COR_POR_ORBITA[orbita])
    limiar = limiares[orbita]
    eixo.axvline(np.log10(limiar), color=COR_REFERENCIA, linestyle="--", linewidth=1.2)
    eixo.annotate(f"P{PERCENTIL_LIMIAR} = {limiar} MW", (np.log10(limiar), 1), xycoords=("data", "axes fraction"),
                  xytext=(4, -12), textcoords="offset points", color=COR_REFERENCIA)
    eixo.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _p: f"{10 ** x:g}"))
    eixo.yaxis.set_major_formatter(mticker.FuncFormatter(milhar))
    eixo.grid(axis="x", visible=False)
    eixo.set_xlabel("FRP (MW, escala log)")
    eixo.set_title(f"Órbita {orbita}")
eixos[0][0].set_ylabel("focos")
figura.suptitle(f"Distribuição do FRP nos meses de treino (até {MES_FIM_TREINO})", x=0.01, ha="left",
                fontweight="bold")
figura.tight_layout()
salvar(figura, "07_distribuicao_frp")
plt.show()

## 8. Arquivos diários (lado do streaming)

Lê os CSVs diários direto do bucket, sem criar tabela, e confere se eles têm o mesmo formato, os mesmos satélites e uma parcela sem FRP parecida com a do histórico. `fora_da_tabela` conta os focos de satélites que não aparecem na Bronze: o streaming aceita esses focos, mas não consegue dizer se são intensos.

In [ ]:
externa = bigquery.ExternalConfig("CSV")
externa.source_uris = [URIS_DIARIOS]
externa.schema = [bigquery.SchemaField(coluna, "STRING") for coluna in COLUNAS_INPE]
externa.options.skip_leading_rows = 1
externa.options.allow_quoted_newlines = True
externa.options.encoding = ENCODING_CSV

try:
    por_dia = consultar(f"""
    WITH orbitas AS ({sql_orbitas()})
    SELECT
      DATE({DATA_HORA}) AS dia,
      COUNT(*) AS focos,
      ROUND(100 * COUNTIF({ausente('d.frp')}) / COUNT(*), 1) AS pct_sem_frp,
      COUNTIF(o.satelite IS NULL) AS fora_da_tabela,
      STRING_AGG(DISTINCT TRIM(d.satelite) ORDER BY TRIM(d.satelite)) AS satelites
    FROM diarios AS d
    LEFT JOIN orbitas AS o ON TRIM(d.satelite) = o.satelite
    GROUP BY dia
    ORDER BY dia
    """, tabelas_externas={"diarios": externa})
except Exception as erro:
    por_dia = pd.DataFrame()
    print(f"Não foi possível ler os diários ({type(erro).__name__}): {erro}")
    print("Confira se o crawler já baixou os arquivos diários e se eles têm as 16 colunas.")

if not por_dia.empty:
    RESUMO["dias_streaming"] = len(por_dia)
    RESUMO["focos_streaming"] = int(por_dia["focos"].sum())
    RESUMO["pct_sem_frp_diarios"] = round(float(np.average(por_dia["pct_sem_frp"].astype(float), weights=por_dia["focos"].astype(float))), 1)
    RESUMO["fora_da_tabela_diarios"] = int(por_dia["fora_da_tabela"].sum())
    print(f"{len(por_dia)} dias, {milhar(RESUMO['focos_streaming'])} focos, {RESUMO['pct_sem_frp_diarios']}% sem FRP")
    display(por_dia)

In [ ]:
if not por_dia.empty:
    figura, eixo = plt.subplots(figsize=(10, 3.8))
    dias = pd.to_datetime(por_dia["dia"].astype(str))
    eixo.plot(dias, por_dia["pct_sem_frp"], color=COR_PRINCIPAL, linewidth=2, marker="o", markersize=4)
    eixo.axhline(RESUMO["pct_sem_frp_geral"], color=COR_REFERENCIA, linestyle="--", linewidth=1.2)
    eixo.annotate(f"histórico: {RESUMO['pct_sem_frp_geral']}%", (dias.iloc[0], RESUMO["pct_sem_frp_geral"]),
                  xytext=(0, 5), textcoords="offset points", color=COR_REFERENCIA)
    eixo.set_ylim(0, 100)
    eixo.set_ylabel("% dos focos sem FRP")
    eixo.set_title("Focos sem FRP por dia nos arquivos diários (streaming)")
    figura.autofmt_xdate()
    salvar(figura, "08_sem_frp_diarios")
    plt.show()

## 9. Resumo para a equipe

Gera o texto de `docs/data_profile.md`. Ele também fica salvo como `data_profile.md` na pasta desta execução, junto com os gráficos. Depois a pasta é copiada para o bucket, em `gs://<BUCKET>/evidencias/perfil/<execução>/`.

In [ ]:
def lista(valores) -> str:
    return ", ".join(str(v) for v in valores) if valores else "nenhum"


linhas = [
    "# Perfil dos dados",
    "",
    f"Gerado pelo notebook `notebooks/01_data_profile.ipynb` sobre `{TABELA_BRONZE}`.",
    "",
    "## Volume",
    f"- {milhar(RESUMO['total_focos'])} focos em {RESUMO['meses']} meses.",
    f"- Focos fora do mês do próprio arquivo: {RESUMO['fora_do_mes']}.",
    "",
    "## Colunas",
    f"- Colunas com vazio ou -999 (% dos focos): "
    f"{lista(f'{c} ({p}%)' for c, p in RESUMO['colunas_com_ausentes'])}.",
    f"- Colunas numéricas com texto que não converte: {lista(RESUMO['colunas_nao_numericas'])}.",
    f"- Colunas com espaços sobrando: {lista(RESUMO['colunas_com_espacos'])}.",
    f"- Textos com acento quebrado: {RESUMO['acentos_suspeitos']}.",
    f"- Estados distintos: {RESUMO['qtd_estados']}; biomas: {lista(RESUMO['biomas'])}.",
    "",
    "## Unicidade",
    f"- {RESUMO['conclusao_id']}",
    "",
    "## Satélites e FRP",
    f"- Satélites: {lista(RESUMO['satelites'])}.",
    f"- Sem órbita na tabela satelites: {lista(RESUMO['satelites_desconhecidos'])}.",
    f"- Nunca trazem FRP: {lista(RESUMO['satelites_nunca_tem_frp'])}.",
    f"- Focos sem FRP no histórico: {RESUMO['pct_sem_frp_geral']}% (peso da justificativa 1).",
    f"- Limiar candidato P{PERCENTIL_LIMIAR} (treino até {MES_FIM_TREINO}): "
    f"{lista(f'{o} = {v} MW' for o, v in RESUMO['limiares_candidatos'].items())}.",
    "",
    "## Streaming",
    f"- Rejeição estimada pelas regras de validação: até {RESUMO['pct_rejeicao_estimada']}% dos focos.",
]
if "dias_streaming" in RESUMO:
    linhas += [
        f"- Arquivos diários: {RESUMO['dias_streaming']} dias, {milhar(RESUMO['focos_streaming'])} focos.",
        f"- Focos sem FRP nos diários: {RESUMO['pct_sem_frp_diarios']}% (histórico: {RESUMO['pct_sem_frp_geral']}%).",
        f"- Focos dos diários de satélite fora da tabela de órbitas (aceitos, sem contar como intensos): "
        f"{RESUMO['fora_da_tabela_diarios']}.",
    ]
linhas += ["", f"Total lido pelo notebook: {BYTES_LIDOS['total'] / 1024**3:.2f} GB."]

texto = "\n".join(linhas)
with open(os.path.join(PASTA_SAIDA, "data_profile.md"), "w", encoding="utf-8") as arquivo:
    arquivo.write(texto + "\n")
print(texto)

In [ ]:
if ENVIAR_PARA_BUCKET:
    balde = storage.Client(project=PROJECT_ID).bucket(BUCKET)
    arquivos = sorted(os.listdir(PASTA_SAIDA))
    for nome in arquivos:
        balde.blob(f"{PASTA_BUCKET_SAIDA}/{nome}").upload_from_filename(os.path.join(PASTA_SAIDA, nome))
    print(f"{len(arquivos)} arquivos copiados para gs://{BUCKET}/{PASTA_BUCKET_SAIDA}/")
    for nome in arquivos:
        print(f"  {nome}")